# Approximate Inference using Gibbs Sampling

## 1. Imports and Model Setup

In [243]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from pathlib import Path
from pgmpy.inference import VariableElimination
from parameter_inference import load_fitted_model, exact_popularity

In [244]:
SEED = 150

with open("models/parameter_config.json") as f:
    config = json.load(f)

model = load_fitted_model(config["model_path"])

print("Structure:", config["structure"])
print("Estimator:", config["estimator"])
print("Equivalent Sample Size:", config["equivalent_sample_size"])
print("Valid Model:", model.check_model())

Structure: HC start=hand
Estimator: BDeu
Equivalent Sample Size: 20.0
Valid Model: True


In [245]:
queries = config["query_specs"]
popularity_states = config["state_names"]["popularity"]

ve_results = pd.read_csv("models/ve_queries.csv")

ve_results = ve_results[
    ve_results["model_id"] == config["primary_model_id"]
].copy()

print("Number of Queries:", len(queries))
print("Popularity States:", popularity_states)

ve_results.head()

Number of Queries: 80
Popularity States: ['(-0.001, 20.0]', '(20.0, 40.0]', '(40.0, 60.0]', '(60.0, 80.0]', '(80.0, 100.0]']


,model_id,query_id,experiment,evidence_json,train_evidence_count,popularity_state,probability
400,HC start=hand,prior,prior,{},3571,"(-0.001, 20.0]",0.069618
401,HC start=hand,prior,prior,{},3571,"(20.0, 40.0]",0.192982
402,HC start=hand,prior,prior,{},3571,"(40.0, 60.0]",0.343358
403,HC start=hand,prior,prior,{},3571,"(60.0, 80.0]",0.335283
404,HC start=hand,prior,prior,{},3571,"(80.0, 100.0]",0.058758


## 2. Markov Blanket Conditional Probabilities

In [246]:
print("Variables:", list(model.nodes()))
print("Parents of popularity:", model.get_parents("popularity"))
print("Children of popularity:", model.get_children("popularity"))
print("Markov blanket:", model.get_markov_blanket("popularity"))

Variables: ['mode', 'key', 'popularity', 'energy', 'tempo', 'valence', 'danceability', 'acousticness', 'decade', 'duration']
Parents of popularity: ['decade']
Children of popularity: ['acousticness']
Markov blanket: ['decade', 'acousticness']


In [247]:
state_names = {}

for node in model.nodes():
    state_names[node] = model.get_cpds(node).state_names[node]

for node, states in state_names.items():
    print(node, ":", states)

mode : ['0.0', '1.0']
key : ['0.0', '1.0', '10.0', '11.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0']
popularity : ['(-0.001, 20.0]', '(20.0, 40.0]', '(40.0, 60.0]', '(60.0, 80.0]', '(80.0, 100.0]']
energy : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
tempo : ['fast', 'mid', 'slow', 'upbeat']
valence : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
danceability : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
acousticness : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
decade : ['1990s', '2000s', '2010s', '2020s', 'Before 1990']
duration : ['2-3 min', '3-4 min', '4-5 min', '5+ min', '< 2 min']


In [248]:
cpd = model.get_cpds("popularity")

assignment = {}

for node in cpd.variables:
    assignment[node] = state_names[node][0]

print("Assignment:", assignment)
print("Probability:", cpd.get_value(**assignment))

Assignment: {'popularity': '(-0.001, 20.0]', 'decade': '1990s'}
Probability: 0.06037735849056604


In [249]:
def get_cpt_probability(model, node, assignment):
    cpd = model.get_cpds(node)

    values = {
        var: assignment[var]
        for var in cpd.variables
    }

    return cpd.get_value(**values)

In [250]:
def markov_blanket_distribution(model, node, assignment):
    probabilities = []

    for state in state_names[node]:
        current = assignment.copy()
        current[node] = state

        probability = get_cpt_probability(model, node, current)

        for child in model.get_children(node):
            probability *= get_cpt_probability(model, child, current)

        probabilities.append(probability)

    probabilities = np.array(probabilities, dtype=float)

    return probabilities / probabilities.sum()

In [251]:
assignment = {
    node: states[0]
    for node, states in state_names.items()
}

probs = markov_blanket_distribution(
    model, "popularity", assignment
)

for state, prob in zip(popularity_states, probs):
    print(state, ":", prob)

print("Total:", probs.sum())

(-0.001, 20.0] : 0.0516594072597332
(20.0, 40.0] : 0.10799801567380884
(40.0, 60.0] : 0.12389707853899191
(60.0, 80.0] : 0.637233830436836
(80.0, 100.0] : 0.07921166809063009
Total: 1.0


## 3. Gibbs Sampling Implementation

In [252]:
def initialize_assignment(evidence, rng):
    assignment = {}

    for node, states in state_names.items():
        if node in evidence:
            assignment[node] = evidence[node]
        else:
            assignment[node] = rng.choice(states)

    return assignment

In [253]:
rng = np.random.default_rng(SEED)

evidence = {"decade": "1990s"}

assignment = initialize_assignment(evidence, rng)

print(assignment)

{'mode': np.str_('1.0'), 'key': np.str_('0.0'), 'popularity': np.str_('(40.0, 60.0]'), 'energy': np.str_('(0.8, 1.0]'), 'tempo': np.str_('fast'), 'valence': np.str_('(0.6, 0.8]'), 'danceability': np.str_('(0.2, 0.4]'), 'acousticness': np.str_('(0.2, 0.4]'), 'decade': '1990s', 'duration': np.str_('5+ min')}


In [254]:
def gibbs_sampling(model, query, evidence,
                   n_samples=5000, burn_in=500, seed=150):

    rng = np.random.default_rng(seed)

    assignment = initialize_assignment(evidence, rng)

    free_nodes = [
        node for node in model.nodes()
        if node not in evidence
    ]

    counts = {
        state: 0
        for state in state_names[query]
    }

    for iteration in range(n_samples + burn_in):

        for node in free_nodes:
            probs = markov_blanket_distribution(
                model, node, assignment
            )

            assignment[node] = rng.choice(
                state_names[node], p=probs
            )

        if iteration >= burn_in:
            counts[assignment[query]] += 1

    return {
        state: count / n_samples
        for state, count in counts.items()
    }

In [255]:
gibbs_result = gibbs_sampling(
    model,
    query="popularity",
    evidence={"decade": "1990s"},
    n_samples=1000,
    burn_in=200,
    seed=SEED
)

for state, probability in gibbs_result.items():
    print(state, ":", probability)

print("Total:", sum(gibbs_result.values()))

(-0.001, 20.0] : 0.052
(20.0, 40.0] : 0.131
(40.0, 60.0] : 0.185
(60.0, 80.0] : 0.568
(80.0, 100.0] : 0.064
Total: 1.0


## 4. Approximate and Exact Inference Comparison

In [256]:
inference = VariableElimination(model)

exact_result = inference.query(
    variables=["popularity"],
    evidence={"decade": "1990s"},
    show_progress=False
)

print(exact_result)

+----------------------------+-------------------+
| popularity                 |   phi(popularity) |
+============================+===================+
| popularity((-0.001, 20.0]) |            0.0604 |
+----------------------------+-------------------+
| popularity((20.0, 40.0])   |            0.1358 |
+----------------------------+-------------------+
| popularity((40.0, 60.0])   |            0.1783 |
+----------------------------+-------------------+
| popularity((60.0, 80.0])   |            0.5604 |
+----------------------------+-------------------+
| popularity((80.0, 100.0])  |            0.0651 |
+----------------------------+-------------------+


In [257]:
exact_probs = {
    state: float(exact_result.get_value(popularity=state))
    for state in popularity_states
}

comparison = pd.DataFrame({
    "Popularity": popularity_states,
    "Exact": [exact_probs[s] for s in popularity_states],
    "Gibbs": [gibbs_result[s] for s in popularity_states]
})

comparison["Absolute Error"] = abs(
    comparison["Exact"] - comparison["Gibbs"]
)

comparison

,Popularity,Exact,Gibbs,Absolute Error
0,"(-0.001, 20.0]",0.060377,0.052,0.008377
1,"(20.0, 40.0]",0.135849,0.131,0.004849
2,"(40.0, 60.0]",0.178302,0.185,0.006698
3,"(60.0, 80.0]",0.560377,0.568,0.007623
4,"(80.0, 100.0]",0.065094,0.064,0.001094


In [258]:
mae = comparison["Absolute Error"].mean()

print("Mean Absolute Error:", mae)

Mean Absolute Error: 0.005728301886792446


## 5. Validation Across Multiple Queries

In [259]:
print("Total queries:", len(queries))

for query in queries[:5]:
    print(query)

Total queries: 80
{'query_id': 'prior', 'experiment': 'prior', 'evidence': {}}
{'query_id': 'decade_0', 'experiment': 'decade', 'evidence': {'decade': '1990s'}}
{'query_id': 'decade_0_energy_0', 'experiment': 'energy', 'evidence': {'decade': '1990s', 'energy': '(-0.001, 0.2]'}}
{'query_id': 'decade_0_energy_1', 'experiment': 'energy', 'evidence': {'decade': '1990s', 'energy': '(0.2, 0.4]'}}
{'query_id': 'decade_0_energy_2', 'experiment': 'energy', 'evidence': {'decade': '1990s', 'energy': '(0.4, 0.6]'}}


In [260]:
print("Columns:", ve_results.columns.tolist())

ve_results.head()

Columns: ['model_id', 'query_id', 'experiment', 'evidence_json', 'train_evidence_count', 'popularity_state', 'probability']


,model_id,query_id,experiment,evidence_json,train_evidence_count,popularity_state,probability
400,HC start=hand,prior,prior,{},3571,"(-0.001, 20.0]",0.069618
401,HC start=hand,prior,prior,{},3571,"(20.0, 40.0]",0.192982
402,HC start=hand,prior,prior,{},3571,"(40.0, 60.0]",0.343358
403,HC start=hand,prior,prior,{},3571,"(60.0, 80.0]",0.335283
404,HC start=hand,prior,prior,{},3571,"(80.0, 100.0]",0.058758


In [261]:
def validate_query(query_spec, n_samples=1000,
                   burn_in=200, seed=SEED):

    query_id = query_spec["query_id"]
    experiment = query_spec["experiment"]
    evidence = query_spec["evidence"]

    start = time.perf_counter()

    gibbs_probs = gibbs_sampling(
        model,
        query="popularity",
        evidence=evidence,
        n_samples=n_samples,
        burn_in=burn_in,
        seed=seed
    )

    runtime = time.perf_counter() - start

    rows = []

    for state in popularity_states:
        rows.append({
            "query_id": query_id,
            "experiment": experiment,
            "popularity_state": state,
            "gibbs_probability": gibbs_probs[state],
            "runtime": runtime,
            "n_samples": n_samples
        })

    return rows

In [ ]:
validation_rows = []

for i, query_spec in enumerate(queries):

    rows = validate_query(
        query_spec,
        n_samples=1000,
        burn_in=200,
        seed=SEED + i
    )

    validation_rows.extend(rows)

    print(f"Completed {i + 1}/{len(queries)}:",
          query_spec["query_id"])

gibbs_results = pd.DataFrame(validation_rows)

print("Total rows:", len(gibbs_results))

Completed 1/80: prior
Completed 2/80: decade_0
Completed 3/80: decade_0_energy_0
Completed 4/80: decade_0_energy_1
Completed 5/80: decade_0_energy_2
Completed 6/80: decade_0_energy_3
Completed 7/80: decade_0_energy_4
Completed 8/80: decade_0_acousticness_0
Completed 9/80: decade_0_acousticness_1
Completed 10/80: decade_0_acousticness_2
Completed 11/80: decade_0_acousticness_3
Completed 12/80: decade_0_acousticness_4
Completed 13/80: decade_0_joint_0_0
Completed 14/80: decade_0_joint_0_4
Completed 15/80: decade_0_joint_4_0
Completed 16/80: decade_0_joint_4_4
Completed 17/80: decade_1
Completed 18/80: decade_1_energy_0
Completed 19/80: decade_1_energy_1
Completed 20/80: decade_1_energy_2
Completed 21/80: decade_1_energy_3
Completed 22/80: decade_1_energy_4
Completed 23/80: decade_1_acousticness_0
Completed 24/80: decade_1_acousticness_1
Completed 25/80: decade_1_acousticness_2
Completed 26/80: decade_1_acousticness_3
Completed 27/80: decade_1_acousticness_4
Completed 28/80: decade_1_join

In [ ]:
validation_results = gibbs_results.merge(
    ve_results[
        ["query_id", "popularity_state", "probability"]
    ],
    on=["query_id", "popularity_state"],
    how="inner",
    validate="one_to_one"
)

validation_results = validation_results.rename(
    columns={"probability": "exact_probability"}
)

validation_results["absolute_error"] = abs(
    validation_results["exact_probability"] -
    validation_results["gibbs_probability"]
)

print("Matched rows:", len(validation_results))

validation_results.head()

Matched rows: 400


,query_id,experiment,popularity_state,gibbs_probability,runtime,n_samples,exact_probability,absolute_error
0,prior,prior,"(-0.001, 20.0]",0.073,1.174196,1000,0.069618,0.003382
1,prior,prior,"(20.0, 40.0]",0.198,1.174196,1000,0.192982,0.005018
2,prior,prior,"(40.0, 60.0]",0.289,1.174196,1000,0.343358,0.054358
3,prior,prior,"(60.0, 80.0]",0.382,1.174196,1000,0.335283,0.046717
4,prior,prior,"(80.0, 100.0]",0.058,1.174196,1000,0.058758,0.000758


In [ ]:
print("Queries tested:",
      validation_results["query_id"].nunique())

print("Overall MAE:",
      validation_results["absolute_error"].mean())

print("Maximum absolute error:",
      validation_results["absolute_error"].max())

Queries tested: 80
Overall MAE: 0.007978750815730873
Maximum absolute error: 0.05435839598997494


In [ ]:
experiment_summary = validation_results.groupby(
    "experiment"
).agg(
    queries=("query_id", "nunique"),
    mean_absolute_error=("absolute_error", "mean"),
    max_absolute_error=("absolute_error", "max")
)

experiment_summary

,queries,mean_absolute_error,max_absolute_error
experiment,,,
acousticness,25,0.007138,0.032271
decade,5,0.009680,0.037320
energy,25,0.008406,0.031197
energy_acousticness,20,0.006997,0.036635
partial,4,0.009829,0.026084
prior,1,0.022047,0.054358


## 6. Convergence Analysis

In [ ]:
sample_sizes = [100, 500, 1000, 5000, 10000]

convergence_queries = [
    queries[0],
    queries[1],
    queries[2]
]

for query_spec in convergence_queries:
    print(query_spec)